# Study 807 — Salience-Theory Returns — the teardown

The per-leg splits, the Newey-West spread *t*, the pooled Welch book test, the 1,000-permutation placebo, the two-era robustness cut, the costed timer, and the 20-seed synthetic control.

In [1]:
R = {'start': '2010-01-04', 'end': '2026-06-30', 'n_names': 50, 'n_days': 4125, 'fingerprint': '357fd262912f', 'spread_bps': -1.0, 't_nw': -0.78, 't_1s': -0.74, 'lo_bps': 7.49, 'hi_bps': 8.49, 'welch_t': -0.37, 'gross_sharpe': -0.18, 'placebo_obs': -1.0, 'placebo_mean': 0.009, 'placebo_sd': 0.896, 'placebo_p': 0.867, 'placebo_sigma': -1.13, 'placebo_draws': 1000, 'era_early_bps': -0.8, 'era_early_t': -0.6, 'era_early_n': 1991, 'era_late_bps': -1.19, 'era_late_t': -0.55, 'era_late_n': 2134, 'timer_1_gross': -1.0, 'timer_1_cost': 2.14, 'timer_1_net': -3.14, 'timer_1_t': -2.32, 'timer_5_gross': -1.0, 'timer_5_cost': 10.14, 'timer_5_net': -11.14, 'timer_5_t': -8.23, 'null_mean_t': -0.26, 'null_sd_t': 0.91, 'null_fire': 1, 'planted_t': 4.31, 'planted_welch': 4.46}

## The headline — long-low-ST / short-high-ST spread

Daily equal-weight bottom-30% (low ST) minus top-30% (high ST) spread.

In [2]:
print(f"spread        : {R['spread_bps']:+.2f} bps/day  NW(10) t = {R['t_nw']:+.2f}  "
      f"one-sample t = {R['t_1s']:+.2f}")
print(f"books         : low-ST {R['lo_bps']:+.2f} vs high-ST {R['hi_bps']:+.2f} bps "
      f"(Welch t = {R['welch_t']:+.2f})")
print(f"gross Sharpe  : {R['gross_sharpe']:.2f} (before cost)")

spread        : -1.00 bps/day  NW(10) t = -0.78  one-sample t = -0.74
books         : low-ST +7.49 vs high-ST +8.49 bps (Welch t = -0.37)
gross Sharpe  : -0.18 (before cost)


## Placebo — column-permute the forward returns (1,000 permutations)

In [3]:
print(f"observed {R['placebo_obs']:+.2f} bps vs placebo mean {R['placebo_mean']:+.3f} "
      f"(sd {R['placebo_sd']:.3f}) -> p = {R['placebo_p']:.5f} (observed ~{R['placebo_sigma']:+.2f} sigma)")

observed -1.00 bps vs placebo mean +0.009 (sd 0.896) -> p = 0.86700 (observed ~-1.13 sigma)


## Robustness — two eras (split 2018-01-01)

In [4]:
print(f"2010-2017 (n={R['era_early_n']}): {R['era_early_bps']:+.2f} bps  NW t = {R['era_early_t']:+.2f}")
print(f"2018-2026 (n={R['era_late_n']}): {R['era_late_bps']:+.2f} bps  NW t = {R['era_late_t']:+.2f}")

2010-2017 (n=1991): -0.80 bps  NW t = -0.60
2018-2026 (n=2134): -1.19 bps  NW t = -0.55


## The timer — can you get paid for it?

2 sides × one-way cost × NAV per day on the long-short book; short pays 50 bps/yr borrow.

In [5]:
for tag,g,c,n,t in [('1 bp',R['timer_1_gross'],R['timer_1_cost'],R['timer_1_net'],R['timer_1_t']),
                    ('5 bps',R['timer_5_gross'],R['timer_5_cost'],R['timer_5_net'],R['timer_5_t'])]:
    print(f"{tag:>5} one-way: gross {g:+.2f} -> net {n:+.2f} bps/day (cost {c:.2f}/day, t={t:+.2f})")

 1 bp one-way: gross -1.00 -> net -3.14 bps/day (cost 2.14/day, t=-2.32)
5 bps one-way: gross -1.00 -> net -11.14 bps/day (cost 10.14/day, t=-8.23)


## Synthetic positive control — the machinery is unbiased

Live: the detector must NOT fire on the null and must recover a planted relation.

In [6]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
import numpy as np
from salience_theory import data, strategy as st
null_t = np.array([st.synthetic_detect(data.synthetic_panel(edge=0.0, seed=807+s, n_assets=40, n_days=1200))['t_nw'] for s in range(8)])
print(f"null (edge=0), 8 seeds: NW t mean {null_t.mean():+.2f} (sd {null_t.std(ddof=1):.2f}), |t|>=2 in {(abs(null_t)>=2).sum()}/8")
planted = st.synthetic_detect(data.synthetic_panel(edge=0.0016, seed=807, n_assets=40, n_days=1500))
print(f"planted (edge=0.0016): NW t = {planted['t_nw']:+.2f}, Welch t = {planted['welch_t']:+.2f}")

null (edge=0), 8 seeds: NW t mean -0.15 (sd 0.99), |t|>=2 in 1/8


planted (edge=0.0016): NW t = +4.31, Welch t = +4.46


## Verdict

- **Signal — None.** The Cosemans-Frehen salience-theory premium does **not** show up on 50 liquid US mega-caps: the long-low-ST / short-high-ST spread is **-1.00 bps/day** (NW *t* = **-0.78**) — indistinguishable from zero, flat in both eras (*t* = -0.60 / -0.55), ~1.1σ from the permutation-null centre (p = 0.87). The synthetic control recovers a *planted* relation cleanly (*t* = +4.31), so the flat tape is a real null, not machinery. The effect is a broad-cross-section (small-cap-inclusive) phenomenon; survivorship on mega-caps kills what little is left.
- **Tradability — Mirage.** There is no gross edge to harvest (-1.00 bps/day), and the book bleeds net: at 1 bp one-way the 2.14 bps/day friction makes it **-3.14 bps/day** (*t* = -2.32); at 5 bps **-11.14 bps/day**.